# 01 — Advanced Retrieval Strategies

**Track:** Intermediate · **Stage:** Retrieval Engineering

Retrieval quality depends on matching the query type to the retrieval signal. Exact identifiers (like **AX-774-B**) are often lexical problems. Abstract concepts (like **"supplier for the Atlas database"**) are semantic problems. 

In this deep dive, you will build and evaluate a controlled retrieval-engineering experiment comparing:
1. **Dense Retrieval:** Semantic similarity using vector embeddings.
2. **Sparse Retrieval:** Keyword matching using BM25.
3. **Hybrid Retrieval:** Fusing sparse and dense scores using Reciprocal Rank Fusion (RRF).
4. **Query Expansion:** Using LLMs to rewrite user queries before retrieval.

We will run a benchmark against a synthetic 27-chunk enterprise corpus using a labelled 23-query dataset.

## 1. Setup and Ingestion

We will use LangChain's specialized retrievers to build these pipelines.

In [ ]:
# Requirements:
# !pip install langchain langchain-chroma langchain-huggingface chromadb rank_bm25 python-dotenv pandas

import os
import json
import time
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever, MultiQueryRetriever
from langchain_openai import ChatOpenAI

def print_results(results):
    for i, doc in enumerate(results):
        print(f"[{i+1}] [{doc.metadata['id']}] Source: {doc.metadata['source']}")
        print(f"{doc.page_content}\n")


## 2. Mock Enterprise Data

We load our synthetic enterprise corpus and our labelled evaluation dataset.

In [ ]:
with open("data/corpus.json", "r") as f:
    raw_corpus = json.load(f)
    
corpus = [
    Document(
        page_content=item["content"],
        metadata={"id": item["id"], "source": item["metadata"]["source"]}
    ) for item in raw_corpus
]

with open("data/eval_dataset.json", "r") as f:
    eval_dataset = json.load(f)
    
print(f"Loaded {len(corpus)} documents.")
print(f"Loaded {len(eval_dataset)} evaluation queries.")


## 3. Dense Retrieval (Semantic Baseline)

Dense retrieval converts text to dense vectors. It is excellent at understanding meaning (e.g. "supplier" = "vendor") but can struggle with rare identifiers, codes, exact strings, and distinctions not well represented by the embedding model.

*Note: We use `all-MiniLM-L6-v2` as a teaching baseline. Production systems often use larger or fine-tuned embeddings.*

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(corpus, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("--- Dense Retrieval: Semantic Paraphrase ---")
print_results(dense_retriever.invoke("Who supplies our vector DB?"))

print("--- Dense Retrieval: Exact Identifier (May Struggle) ---")
print_results(dense_retriever.invoke("What voltage does AX-774-B require?"))


## 4. Sparse Retrieval (BM25 Lexical Baseline)

BM25 ranks documents using lexical term statistics. It can struggle when relevant documents use substantially different vocabulary from the query, but exact lexical evidence is often exactly what production retrieval needs.

In [ ]:
bm25_retriever = BM25Retriever.from_documents(corpus)
bm25_retriever.k = 3

print("--- Sparse Retrieval: Exact Identifier ---")
print_results(bm25_retriever.invoke("What voltage does AX-774-B require?"))

print("--- Sparse Retrieval: Semantic Paraphrase (May Struggle) ---")
print_results(bm25_retriever.invoke("Who supplies our vector DB?"))


## 5. Hybrid Retrieval (Rank Fusion)

Hybrid retrieval combines complementary candidate lists. It can improve robustness on heterogeneous workloads but should justify its additional complexity through evaluation.

We use **Reciprocal Rank Fusion (RRF)** to combine the results because BM25 scores and dense Cosine Similarity scores exist on completely different, incompatible scales. 

We can pass `weights` to the `EnsembleRetriever` to favor one retriever over another. **These are hyperparameters to evaluate, not fixed rules.**

In [ ]:
# 0.5 / 0.5 equal weighting
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, bm25_retriever], 
    weights=[0.5, 0.5]
)

print("--- Hybrid Retrieval: Exact Identifier ---")
print_results(hybrid_retriever.invoke("What voltage does AX-774-B require?"))


### Candidate Overlap Analysis

Let's see what unique candidates BM25 and Dense retrieval contribute to a mixed query.

In [ ]:
query = "Did Project Helios reduce Q1 2026 costs?"

dense_docs = {d.metadata["id"] for d in dense_retriever.invoke(query)}
bm25_docs = {d.metadata["id"] for d in bm25_retriever.invoke(query)}

print(f"Dense docs: {dense_docs}")
print(f"BM25 docs:  {bm25_docs}")
print(f"Intersection (Both found): {dense_docs.intersection(bm25_docs)}")
print(f"Union (Total candidates): {len(dense_docs.union(bm25_docs))}")


## 6. First-Stage Candidate Budget ($k$)

Retrieval is a candidate-generation budget. If a document is not in the top $k$, a later reranker cannot save it. Let's see how increasing $k$ affects latency and candidate counts for BM25.

In [ ]:
query = "Business Continuity Plan failover time"

for k_val in [1, 3, 5, 10]:
    temp_retriever = BM25Retriever.from_documents(corpus)
    temp_retriever.k = k_val
    
    start = time.time()
    docs = temp_retriever.invoke(query)
    latency = time.time() - start
    
    print(f"k={k_val:<2} | Retrieved {len(docs):<2} docs | Latency: {latency*1000:.2f}ms")


## 7. Query Expansion (Multi-Query)

Query expansion can improve recall for some query classes but can also add latency, redundancy, or semantic drift. We use an LLM to generate multiple variants of the user's question, retrieve documents for all variants, and deduplicate the results.

In [ ]:
# Set up a real LLM (requires OPENAI_API_KEY)
mq_retriever = None
try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    mq_retriever = MultiQueryRetriever.from_llm(retriever=dense_retriever, llm=llm)
    
    import logging
    logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)
    
    print("--- Multi-Query Retrieval ---")
    mq_results = mq_retriever.invoke("Who supplies our vector DB?")
    print(f"\nRetrieved {len(mq_results)} unique documents across all variants.")
except Exception as e:
    print("LLM Setup failed. Please ensure OPENAI_API_KEY is in your .env file.")
    print(e)


### Query Drift Danger

Variants are search hypotheses, not authoritative interpretations of the user's intent. **Never expand authorization constraints!** Wording may change, but security filters must remain identical across all variants.

Look at how an LLM might mistakenly expand a specific operational query into a broader, unrelated topic:

In [ ]:
original_query = "What is the approved rollback procedure for payments?"
bad_expansion = "How can engineers quickly restart the payments infrastructure?"

print("Original query retrieves the rollback runbook:")
print([d.metadata["id"] for d in dense_retriever.invoke(original_query)])

print("\nBad expansion retrieves the restart runbook (Drift!):")
print([d.metadata["id"] for d in dense_retriever.invoke(bad_expansion)])


## 8. Benchmark Evaluation

Aggregate metrics are not enough. We will evaluate our strategies across specific **query slices** to see where they fail.

In [ ]:
def recall_at_k(retrieved_docs, relevant_ids):
    if not relevant_ids: return 1.0 # True negative
    retrieved_ids = [doc.metadata["id"] for doc in retrieved_docs]
    hits = sum(1 for rid in relevant_ids if rid in retrieved_ids)
    return hits / len(relevant_ids)

def reciprocal_rank(retrieved_docs, relevant_ids):
    if not relevant_ids: return 1.0
    retrieved_ids = [doc.metadata["id"] for doc in retrieved_docs]
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant_ids:
            return 1.0 / rank
    return 0.0

def evaluate_retriever(retriever, dataset, name):
    results = []
    total_latency = 0
    
    for item in dataset:
        start = time.time()
        docs = retriever.invoke(item["query"])
        latency = time.time() - start
        total_latency += latency
        
        results.append({
            "strategy": name,
            "slice": item["slice"],
            "query": item["query"],
            "recall": recall_at_k(docs, item["relevant_ids"]),
            "mrr": reciprocal_rank(docs, item["relevant_ids"])
        })
        
    print(f"{name:15} | Avg Latency: {(total_latency/len(dataset))*1000:.1f}ms")
    return pd.DataFrame(results)


In [ ]:
# Run the benchmark
print("Running benchmark...")
df_dense = evaluate_retriever(dense_retriever, eval_dataset, "Dense (k=3)")
df_bm25 = evaluate_retriever(bm25_retriever, eval_dataset, "BM25 (k=3)")
df_hybrid = evaluate_retriever(hybrid_retriever, eval_dataset, "Hybrid (0.5/0.5)")
    
if mq_retriever:
    df_mq = evaluate_retriever(mq_retriever, eval_dataset, "MultiQuery")
    df_all = pd.concat([df_dense, df_bm25, df_hybrid, df_mq])
else:
    df_all = pd.concat([df_dense, df_bm25, df_hybrid])


# Group by slice
summary = df_all.groupby(["slice", "strategy"])[["recall", "mrr"]].mean().unstack()
display(summary)


## 9. Failure Analysis & Decision Summary

Review the results above. You should observe:
1. **BM25 wins on identifiers**: Exact strings (like AX-774-B) and Acronyms strongly favor lexical search.
2. **Dense wins on semantic paraphrase**: Wording differences (lawyers vs data retention policy) are bridged by vector embeddings.
3. **Hybrid balances the two**: It provides robust performance across mixed slices.
4. **Hard Negatives**: Both struggle when the exact text overlaps but constraints differ (e.g., legacy cooling controller).

### Decision Guide
- **exact terms missed** → lexical/sparse
- **semantic paraphrase miss** → dense
- **mixed population** → hybrid
- **poor candidate ordering** → reranker
- **single-vector weakness** → late interaction / multi-representation
